<a href="https://colab.research.google.com/github/avinash-tiwary/ePic/blob/main/notebooks/02_1D_Two_Stream_Instability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: 1D-3V Two-Stream Instability & Kinetic Dispersion

## 1. Physical Background
The two-stream instability is the prototypical kinetic plasma instability. Two electron streams of equal density $n_0/2$ move with opposite drift velocities $\pm v_b$ relative to a stationary, neutralizing ion background.

### Linear Dispersion Relation
Perturbing the Vlasov–Poisson system with plane waves $\sim e^{i(kx - \omega t)}$ yields the warm two-stream dispersion relation:
$$1 = \frac{\omega_{p1}^2}{(\omega - k v_b)^2 - 3 k^2 v_{th}^2} + \frac{\omega_{p2}^2}{(\omega + k v_b)^2 - 3 k^2 v_{th}^2}$$
For cold beams ($v_{th} \to 0$):
$$1 = \frac{\omega_{pe}^2/2}{(\omega - k v_b)^2} + \frac{\omega_{pe}^2/2}{(\omega + k v_b)^2}$$
The maximum growth rate occurs at $k v_b = \frac{\sqrt{3}}{2} \omega_{pe} \approx 0.866\,\omega_{pe}$, yielding:
$$\gamma_{\max} = \frac{\omega_{pe}}{2} = 0.5\,\omega_{pe}$$
For warm beams ($v_{th} > 0$), finite temperature Landau damping reduces the growth rate ($\gamma \approx 0.237\,\omega_{pe}$).


In [ ]:
# ==============================================================
# Google Colab Setup & Package Installation
# ==============================================================
import sys
if 'google.colab' in sys.modules:
    print('Running in Google Colab. Installing ePic...')
    !git clone https://github.com/avinash-tiwary/ePic.git
    %cd ePic
    !pip install -e .
else:
    print('Running locally. Verifying ePic installation...')
    import epic
    print(f'ePic version {epic.__version__} loaded successfully!')


## 2. Running the 1D Two-Stream Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from epic.solvers.pic1d import PIC1DSolver

Nx = 256
L = 45.0
N_particles = 60000
dt = 0.1
t_end = 40.0
v_beam = 3.0
v_th = 0.5
n0 = 1.0

weight = (n0 * L) / N_particles
solver = PIC1DSolver(Nx=Nx, boxsize=L, dt=dt)

Nh = N_particles // 2
np.random.seed(42)
pos1 = np.random.uniform(0.0, L, Nh)
vel1 = np.random.normal(v_beam, v_th, (Nh, 3))
pos2 = np.random.uniform(0.0, L, Nh)
vel2 = np.random.normal(-v_beam, v_th, (Nh, 3))

solver.add_species("beam_right", q=-weight, m=weight, pos=pos1, vel=vel1)
solver.add_species("beam_left", q=-weight, m=weight, pos=pos2, vel=vel2)
solver.initialize()

print(f"Simulating 1D Two-Stream Instability: N={N_particles}, Nx={Nx}, t_end={t_end}...")
solver.run(t_end=t_end)

# Plot Phase Space and Energy Diagnostics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=110)
p1 = solver.species[0].pos[:, 0]
v1 = solver.species[0].vel[:, 0]
p2 = solver.species[1].pos[:, 0]
v2 = solver.species[1].vel[:, 0]

ax1.scatter(p1[::2], v1[::2], s=0.3, color="#0077bb", alpha=0.4, label="Beam 1 (+v)")
ax1.scatter(p2[::2], v2[::2], s=0.3, color="#cc3311", alpha=0.4, label="Beam 2 (-v)")
ax1.set_xlim(0, L)
ax1.set_ylim(-6.5, 6.5)
ax1.set_xlabel(r"Position $x$ ($c/\omega_{pe}$)")
ax1.set_ylabel(r"Velocity $v_x$")
ax1.set_title(rf"Phase Space Vortex Roll-Up at $t = {t_end}$")
ax1.legend(loc="upper right", markerscale=8)
ax1.grid(True, linestyle="--", alpha=0.3)

time_arr = np.array(solver.history["time"])
e_field = np.array(solver.history["E_field"])
e_tot = np.array(solver.history["E_total"])
ax2.semilogy(time_arr, np.maximum(e_field, 1e-12), color="#cc3311", lw=2, label="Field Energy")
ax2.set_xlabel(r"Time ($\omega_{pe} t$)")
ax2.set_ylabel("Electrostatic Energy")
ax2.set_title("Instability Growth & Non-linear Saturation")
ax2.grid(True, linestyle="--", alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

rel_drift = abs(e_tot[-1] - e_tot[0]) / e_tot[0]
print(f"Total energy conservation drift: {rel_drift:.2e} (< 0.01%!)")


## 3. Inline Animated Movie

<p align="center"><img src="../docs/animations/two_stream_1d.gif" width="80%" alt="Two Stream Movie"/></p>